# Sessions 35-36: 프로젝트 학습 수행

## 🎯 모델 학습 실행

이 노트북에서는 수집/정제한 데이터를 사용하여 **LoRA/QLoRA 기반 파인튜닝**을 수행합니다.

### 학습 파이프라인
```
환경 설정 → 모델 로드 → 데이터 로드 → LoRA 설정 → 학습 → (DPO) → 분석 → 저장
```

### 사전 준비
- `33_project_data_pipeline.ipynb`에서 생성한 `train.json`, `eval.json`
- `project_plan.json`에 저장된 모델 설정

### RTX 4060 안전 기본값
| 파라미터 | 안전값 | 설명 |
|---------|--------|------|
| batch_size | 1~2 | OOM 방지 |
| gradient_accumulation_steps | 8 | 실효 배치=8~16 |
| max_seq_length | 1024 | VRAM 절약 |
| fp16 / bf16 | fp16 | Turing/Ada 호환 |
| gradient_checkpointing | True | VRAM 절약 |

In [ ]:
# =====================================================
# 📦 필수 패키지 임포트
# =====================================================

import torch
import json
import os
from datetime import datetime

# GPU 확인
print(f"🖥️ PyTorch: {torch.__version__}")
print(f"🎮 CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"🎮 VRAM: {vram:.1f}GB")
    print(f"🎮 현재 사용량: {torch.cuda.memory_allocated(0)/1024**3:.2f}GB")

In [ ]:
# =====================================================
# 📋 프로젝트 설정 로드
# =====================================================

PROJECT_DIR = "./my_project"  # TODO: 프로젝트 디렉토리

with open(f"{PROJECT_DIR}/project_plan.json", "r", encoding="utf-8") as f:
    project_plan = json.load(f)

print(f"📋 프로젝트: {project_plan['project_name']}")
print(f"🤖 모델: {project_plan['model_config']['base_model']}")
print(f"🔧 방법: {project_plan['model_config']['method']}")

---
## 1️⃣ 환경 설정 및 모델 로드

> 💡 **Hint**: QLoRA를 사용하면 7B 모델도 RTX 4060에서 학습 가능합니다. 4bit 양자화로 모델을 로드합니다.

In [ ]:
# =====================================================
# 🤖 모델 로드
# TODO: 모델 이름과 로드 방식을 선택하세요
# =====================================================

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# TODO: 모델 이름을 설정하세요
MODEL_NAME = project_plan["model_config"]["base_model"]  # 또는 직접 입력
# 예시: "Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"

METHOD = project_plan["model_config"]["method"]  # "LoRA", "QLoRA", "FFT"

print(f"📦 모델 로드 중: {MODEL_NAME}")
print(f"🔧 방법: {METHOD}")

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 모델 로드 (방법론에 따라 다름)
if METHOD == "QLoRA":
    # 4bit 양자화 설정
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
elif METHOD == "LoRA":
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
else:  # FFT
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

print(f"\n✅ 모델 로드 완료")
print(f"📊 파라미터 수: {model.num_parameters()/1e6:.0f}M")
print(f"🎮 GPU 메모리: {torch.cuda.memory_allocated(0)/1024**3:.2f}GB")

---
## 2️⃣ 데이터 로드

이전 노트북에서 준비한 학습 데이터를 로드합니다.

In [ ]:
# =====================================================
# 📂 데이터 로드
# TODO: 데이터 경로를 확인하세요
# =====================================================

from datasets import Dataset

# TODO: 데이터 경로 확인
TRAIN_PATH = f"{PROJECT_DIR}/data/train.json"  # TODO: 학습 데이터 경로
EVAL_PATH = f"{PROJECT_DIR}/data/eval.json"    # TODO: 평가 데이터 경로

# JSON 로드
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_raw = json.load(f)
with open(EVAL_PATH, "r", encoding="utf-8") as f:
    eval_raw = json.load(f)

# HuggingFace Dataset 변환
train_dataset = Dataset.from_list(train_raw)
eval_dataset = Dataset.from_list(eval_raw)

print(f"📊 Train: {len(train_dataset)}개")
print(f"📊 Eval:  {len(eval_dataset)}개")
print(f"\n📝 Train 예시:")
print(train_dataset[0])

In [ ]:
# =====================================================
# 🔧 데이터 포매팅 함수
# TODO: 프로젝트 데이터 형식에 맞게 수정하세요
# =====================================================

def format_instruction(sample):
    """
    데이터를 모델의 chat template에 맞게 포매팅합니다.
    
    TODO: 프로젝트의 시스템 프롬프트와 형식을 수정하세요.
    """
    # TODO: 도메인에 맞는 시스템 프롬프트 작성
    system_msg = "당신은 도움이 되는 AI 어시스턴트입니다."  # TODO: 수정하세요
    
    # Alpaca 형식인 경우
    user_msg = sample["instruction"]
    if sample.get("input"):
        user_msg += f"\n\n{sample['input']}"
    
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": sample["output"]}
    ]
    
    # chat template 적용
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

# 포매팅 적용
train_formatted = train_dataset.map(format_instruction)
eval_formatted = eval_dataset.map(format_instruction)

print("✅ 데이터 포매팅 완료")
print(f"\n📝 포매팅 예시:")
print(train_formatted[0]["text"][:500])

---
## 3️⃣ LoRA/QLoRA 설정

> 💡 **Hint**: LoRA rank(r)가 높을수록 표현력이 좋지만 메모리를 더 사용합니다. RTX 4060에서는 r=16~32가 적당합니다.

In [ ]:
# =====================================================
# 🔧 LoRA 설정
# TODO: 하이퍼파라미터를 조정하세요
# =====================================================

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

if METHOD in ["LoRA", "QLoRA"]:
    # QLoRA인 경우 모델 준비
    if METHOD == "QLoRA":
        model = prepare_model_for_kbit_training(model)
    
    # TODO: LoRA 하이퍼파라미터를 조정하세요
    lora_config = LoraConfig(
        r=16,                   # TODO: LoRA rank (8, 16, 32, 64)
        lora_alpha=32,          # TODO: LoRA alpha (보통 r*2)
        lora_dropout=0.05,      # TODO: Dropout (0.0~0.1)
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=[        # TODO: 타겟 모듈 선택
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            # "gate_proj",       # 추가 시 성능↑, VRAM↑
            # "up_proj",
            # "down_proj",
        ],
    )
    
    # LoRA 적용
    model = get_peft_model(model, lora_config)
    
    # 학습 가능 파라미터 확인
    model.print_trainable_parameters()
    print(f"\n🎮 GPU 메모리: {torch.cuda.memory_allocated(0)/1024**3:.2f}GB")
else:
    print("📌 FFT 모드: 전체 파라미터 학습")
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"학습 가능 파라미터: {trainable/1e6:.0f}M")

---
## 4️⃣ 학습 실행 (SFTTrainer)

### Before: 학습 전 모델 출력 확인
학습 전에 모델의 현재 출력을 확인하여 Before/After 비교를 준비합니다.

In [ ]:
# =====================================================
# 📝 Before: 학습 전 모델 출력 확인
# TODO: 테스트 프롬프트를 도메인에 맞게 수정하세요
# =====================================================

# TODO: 도메인에 맞는 테스트 프롬프트를 작성하세요
test_prompts = [
    "",  # TODO: 테스트 프롬프트 1
    "",  # TODO: 테스트 프롬프트 2
    "",  # TODO: 테스트 프롬프트 3
]

# 시스템 프롬프트
system_msg = "당신은 도움이 되는 AI 어시스턴트입니다."  # TODO: 수정하세요

def generate_response(model, tokenizer, prompt, system_msg="", max_new_tokens=256):
    """모델 응답을 생성합니다."""
    messages = []
    if system_msg:
        messages.append({"role": "system", "content": system_msg})
    messages.append({"role": "user", "content": prompt})
    
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
        )
    
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

# Before 결과 저장
before_results = []
for i, prompt in enumerate(test_prompts):
    if prompt:
        print(f"\n{'='*50}")
        print(f"📝 테스트 {i+1}: {prompt[:50]}...")
        response = generate_response(model, tokenizer, prompt, system_msg)
        before_results.append({"prompt": prompt, "response": response})
        print(f"🤖 [Before] {response[:300]}")

if not any(test_prompts):
    print("⚠️ test_prompts를 작성해주세요.")

In [ ]:
# =====================================================
# 🏋️ SFTTrainer 설정 및 학습 실행
# TODO: 학습 하이퍼파라미터를 조정하세요
# =====================================================

from transformers import TrainingArguments
from trl import SFTTrainer

# 출력 디렉토리
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = f"{PROJECT_DIR}/models/sft_{timestamp}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# TODO: 학습 하이퍼파라미터를 조정하세요
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    
    # === 에포크/스텝 ===
    num_train_epochs=3,                  # TODO: 에포크 수 (1~5)
    # max_steps=500,                     # TODO: 또는 최대 스텝 수로 설정
    
    # === 배치 크기 (RTX 4060 안전값) ===
    per_device_train_batch_size=1,        # TODO: 배치 크기 (1~2)
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,        # TODO: gradient 축적 (4~16)
    
    # === 학습률 ===
    learning_rate=2e-4,                   # TODO: 학습률 (1e-4 ~ 5e-4)
    lr_scheduler_type="cosine",           # 스케줄러
    warmup_ratio=0.1,                     # 워밍업 비율
    
    # === 메모리 최적화 (RTX 4060) ===
    fp16=True,                            # FP16 사용
    gradient_checkpointing=True,          # VRAM 절약
    optim="adamw_torch",                  # 옵티마이저
    
    # === 로깅 ===
    logging_steps=10,                     # 로그 간격
    logging_dir=f"{OUTPUT_DIR}/logs",
    
    # === 평가 ===
    eval_strategy="steps",
    eval_steps=50,                        # 평가 간격
    
    # === 저장 ===
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,                   # 최대 체크포인트 수
    
    # === 기타 ===
    report_to="none",                     # wandb 등 비활성화
    seed=42,
)

# SFTTrainer 생성
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_formatted,
    eval_dataset=eval_formatted,
    processing_class=tokenizer,
    max_seq_length=1024,                  # TODO: 최대 시퀀스 길이 (1024~2048)
)

print(f"✅ 학습 설정 완료")
print(f"📊 Train 데이터: {len(train_formatted)}개")
print(f"📊 Eval 데이터: {len(eval_formatted)}개")
print(f"📊 에포크: {training_args.num_train_epochs}")
print(f"📊 실효 배치 크기: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"📊 총 학습 스텝: ~{len(train_formatted) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * int(training_args.num_train_epochs)}")

In [ ]:
# =====================================================
# 🚀 학습 시작!
# =====================================================

print("🏋️ 학습을 시작합니다...")
print(f"🎮 GPU 메모리 (학습 전): {torch.cuda.memory_allocated(0)/1024**3:.2f}GB")

# 학습 실행
train_result = trainer.train()

# 학습 결과 출력
print(f"\n{'='*50}")
print(f"✅ 학습 완료!")
print(f"{'='*50}")
print(f"📊 Total steps: {train_result.global_step}")
print(f"📊 Training loss: {train_result.training_loss:.4f}")
print(f"⏱️ Training time: {train_result.metrics.get('train_runtime', 0):.0f}초")
print(f"🎮 GPU 메모리 (학습 후): {torch.cuda.memory_allocated(0)/1024**3:.2f}GB")

---
## 5️⃣ (Optional) DPO 학습

SFT 후 **DPO(Direct Preference Optimization)**를 추가로 수행하여 모델을 개선할 수 있습니다.

> ⚠️ DPO를 위해서는 **Preference 데이터** (chosen/rejected 쌍)가 필요합니다.
> 이 단계는 선택사항입니다.

In [ ]:
# =====================================================
# 🔄 (Optional) DPO 학습
# TODO: Preference 데이터가 준비된 경우에만 실행하세요
# =====================================================

USE_DPO = False  # TODO: DPO 사용 시 True로 변경

if USE_DPO:
    from trl import DPOTrainer, DPOConfig
    
    # TODO: Preference 데이터 로드
    # 형식: {"prompt": ..., "chosen": ..., "rejected": ...}
    DPO_DATA_PATH = f"{PROJECT_DIR}/data/preference_data.json"  # TODO: 경로
    
    with open(DPO_DATA_PATH, "r", encoding="utf-8") as f:
        dpo_raw = json.load(f)
    dpo_dataset = Dataset.from_list(dpo_raw)
    
    # DPO 설정
    dpo_config = DPOConfig(
        output_dir=f"{PROJECT_DIR}/models/dpo_{timestamp}",
        num_train_epochs=1,                # TODO: DPO 에포크 (1~2)
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=5e-5,                # TODO: 학습률 (SFT보다 낮게)
        beta=0.1,                          # TODO: DPO beta (0.1~0.5)
        fp16=True,
        gradient_checkpointing=True,
        logging_steps=10,
        report_to="none",
    )
    
    # DPO Trainer
    dpo_trainer = DPOTrainer(
        model=model,
        args=dpo_config,
        train_dataset=dpo_dataset,
        processing_class=tokenizer,
    )
    
    print("🏋️ DPO 학습 시작...")
    dpo_result = dpo_trainer.train()
    print(f"✅ DPO 학습 완료! Loss: {dpo_result.training_loss:.4f}")
else:
    print("⏭️ DPO 학습을 건너뜁니다.")

---
## 6️⃣ 학습 로그 분석

학습 과정의 loss 변화를 분석합니다.

In [ ]:
# =====================================================
# 📈 학습 로그 분석
# =====================================================

import matplotlib.pyplot as plt

# 학습 로그 추출
log_history = trainer.state.log_history

# Train loss 추출
train_steps = [log["step"] for log in log_history if "loss" in log]
train_losses = [log["loss"] for log in log_history if "loss" in log]

# Eval loss 추출
eval_steps = [log["step"] for log in log_history if "eval_loss" in log]
eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]

# 그래프 그리기
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train Loss
axes[0].plot(train_steps, train_losses, "b-", alpha=0.7, label="Train Loss")
axes[0].set_xlabel("Steps")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Eval Loss
if eval_losses:
    axes[1].plot(eval_steps, eval_losses, "r-o", alpha=0.7, label="Eval Loss")
    axes[1].set_xlabel("Steps")
    axes[1].set_ylabel("Loss")
    axes[1].set_title("Evaluation Loss")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "Eval 데이터 없음", ha="center", va="center")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_loss.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n📊 최종 Train Loss: {train_losses[-1]:.4f}")
if eval_losses:
    print(f"📊 최종 Eval Loss: {eval_losses[-1]:.4f}")

In [ ]:
# =====================================================
# 📝 After: 학습 후 모델 출력 확인 (Before/After 비교)
# =====================================================

print("📝 Before/After 비교")
print("="*60)

after_results = []
for i, prompt_data in enumerate(before_results):
    prompt = prompt_data["prompt"]
    before_resp = prompt_data["response"]
    
    # After 생성
    after_resp = generate_response(model, tokenizer, prompt, system_msg)
    after_results.append({"prompt": prompt, "response": after_resp})
    
    print(f"\n{'='*60}")
    print(f"📌 테스트 {i+1}: {prompt[:80]}")
    print(f"-"*60)
    print(f"🔵 [Before]: {before_resp[:300]}")
    print(f"-"*60)
    print(f"🟢 [After]:  {after_resp[:300]}")

# 비교 결과 저장
comparison = {
    "before": before_results,
    "after": after_results,
}
with open(f"{OUTPUT_DIR}/before_after_comparison.json", "w", encoding="utf-8") as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)
print(f"\n💾 비교 결과 저장: {OUTPUT_DIR}/before_after_comparison.json")

---
## 7️⃣ 모델 저장

학습된 모델을 저장합니다.

In [ ]:
# =====================================================
# 💾 모델 저장
# =====================================================

SAVE_DIR = f"{PROJECT_DIR}/models/final_model"

if METHOD in ["LoRA", "QLoRA"]:
    # LoRA 어댑터만 저장
    model.save_pretrained(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)
    print(f"✅ LoRA 어댑터 저장 완료: {SAVE_DIR}")
    
    # (Optional) 병합된 모델 저장
    SAVE_MERGED = True  # TODO: 병합 모델 저장 여부
    if SAVE_MERGED:
        MERGED_DIR = f"{PROJECT_DIR}/models/merged_model"
        print("\n🔧 모델 병합 중...")
        merged_model = model.merge_and_unload()
        merged_model.save_pretrained(MERGED_DIR)
        tokenizer.save_pretrained(MERGED_DIR)
        print(f"✅ 병합 모델 저장 완료: {MERGED_DIR}")
else:
    # FFT: 전체 모델 저장
    model.save_pretrained(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)
    print(f"✅ 전체 모델 저장 완료: {SAVE_DIR}")

# 학습 설정 저장
training_info = {
    "model_name": MODEL_NAME,
    "method": METHOD,
    "epochs": training_args.num_train_epochs,
    "batch_size": training_args.per_device_train_batch_size,
    "grad_accum": training_args.gradient_accumulation_steps,
    "learning_rate": training_args.learning_rate,
    "final_train_loss": train_losses[-1] if train_losses else None,
    "final_eval_loss": eval_losses[-1] if eval_losses else None,
    "train_data_count": len(train_formatted),
    "timestamp": timestamp,
}

with open(f"{SAVE_DIR}/training_info.json", "w", encoding="utf-8") as f:
    json.dump(training_info, f, ensure_ascii=False, indent=2)

print(f"\n📋 학습 정보 저장: {SAVE_DIR}/training_info.json")

In [ ]:
# =====================================================
# 🎮 GPU 메모리 정리
# =====================================================

import gc

# 메모리 해제
del trainer
gc.collect()
torch.cuda.empty_cache()

print(f"🎮 GPU 메모리 정리 완료")
print(f"🎮 현재 사용량: {torch.cuda.memory_allocated(0)/1024**3:.2f}GB")

---
## 📝 정리

### 이번 세션에서 완료한 것
- ✅ 모델 로드 (LoRA/QLoRA/FFT)
- ✅ 학습 데이터 포매팅
- ✅ LoRA 설정 및 학습 실행
- ✅ (Optional) DPO 강화학습
- ✅ 학습 로그 분석 (Loss 그래프)
- ✅ Before/After 비교
- ✅ 모델 저장

### 산출물
- 📁 `my_project/models/final_model/` - LoRA 어댑터 (또는 전체 모델)
- 📁 `my_project/models/merged_model/` - 병합된 모델
- 📁 `my_project/models/sft_*/training_loss.png` - Loss 그래프
- 📁 `my_project/models/sft_*/before_after_comparison.json` - 비교 결과

### 하이퍼파라미터 튜닝 가이드
| 증상 | 해결 방법 |
|------|----------|
| Train loss 안 줄어듦 | learning_rate 올리기 (5e-4), 에포크 늘리기 |
| Eval loss 올라감 (오버피팅) | 에포크 줄이기, dropout 올리기, 데이터 늘리기 |
| OOM 에러 | batch_size=1, max_seq_length=512, gradient_checkpointing=True |
| 품질이 낮음 | 데이터 품질 확인, r값 올리기, 타겟 모듈 추가 |

### 다음 노트북
👉 **35_project_evaluation.ipynb**: 프로젝트 성능 평가 및 반복 개선